# Prompt Swallowing — Spot Check

Compare three configurations:
1. **Trained student + prompt** (should be good — it had both advantages)
2. **Base Qwen3-0.6B + prompt** (baseline — no training, but full prompt)
3. **Trained student WITHOUT prompt** (the actual goal — does it behave as if the prompt were there?)

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

CHECKPOINT_DIR = "../outputs/prompt_swallowing/checkpoint-22000"
BASE_MODEL = "Qwen/Qwen3-0.6B"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load the tools config to build the system prompt
with open("../configs/prompt_swallowing_config.json") as f:
    config = json.load(f)

import sys; sys.path.insert(0, "..")
from src.data.data_generator import build_tool_system_prompt

SYSTEM_PROMPT = build_tool_system_prompt(config["tools"])
print(SYSTEM_PROMPT)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

In [ ]:
# Load base model (untrained Qwen3-0.6B)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.eval()
print("Base model loaded")

In [ ]:
# Load trained student (base + LoRA adapter from checkpoint)
trained_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
trained_model = PeftModel.from_pretrained(trained_model, CHECKPOINT_DIR)
trained_model.eval()
print("Trained student loaded from", CHECKPOINT_DIR)

In [ ]:
def generate_response(model, messages, max_new_tokens=256):
    """Generate a response given a list of chat messages."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    # Decode only the generated portion
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
test_queries = [
    "What's 347 * 28?",
    "What's the weather like in Tokyo?",
    "Search for information about quantum computing",
    "Calculate 1024 / 16 + 55",
    "Check weather for Berlin",
]

## 1. Trained student WITH prompt

In [ ]:
print("=" * 70)
print("TRAINED MODEL + SYSTEM PROMPT")
print("=" * 70)
for q in test_queries:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    response = generate_response(trained_model, messages)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 70)

## 2. Base Qwen3-0.6B WITH prompt (untrained baseline)

In [ ]:
print("=" * 70)
print("BASE MODEL + SYSTEM PROMPT (no training)")
print("=" * 70)
for q in test_queries:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    response = generate_response(base_model, messages)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 70)

## 3. Trained student WITHOUT prompt (the actual test)

In [ ]:
print("=" * 70)
print("TRAINED MODEL — NO SYSTEM PROMPT")
print("=" * 70)
for q in test_queries:
    messages = [
        {"role": "user", "content": q},
    ]
    response = generate_response(trained_model, messages)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 70)

## Side-by-side comparison

In [ ]:
for q in test_queries:
    print("=" * 70)
    print(f"QUERY: {q}")
    print("=" * 70)

    msgs_with_prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    msgs_no_prompt = [
        {"role": "user", "content": q},
    ]

    r1 = generate_response(trained_model, msgs_with_prompt)
    r2 = generate_response(base_model, msgs_with_prompt)
    r3 = generate_response(trained_model, msgs_no_prompt)

    print(f"  [trained+prompt]   {r1[:200]}")
    print(f"  [base+prompt]      {r2[:200]}")
    print(f"  [trained-no-prompt] {r3[:200]}")
    print()